In [1]:
import random
from functools import partial
from omegaconf import DictConfig
import os
import torch
import rasterio
import numpy as np
from pathlib import Path
import h5py
from typing import Dict, Any
import matplotlib.pyplot as plt
from omegaconf import OmegaConf

from dataloader_CIRCA.datasets import CIRCA_ADAPTED2UTILISE_Dataset
from typing import Any, Dict, Literal
from sklearn.metrics import r2_score
import math
import torch
import torchgeometry as tgm
from prodict import Prodict
from torch import Tensor
from lib.data_utils import extract_sample
from lib.data_utils import seed_worker, pad_collate
from lib import config_utils
from lib import data_utils
from lib import visutils # gallery / apply_brightness_factor / sequence2gallery 
from lib.eval_tools import (
    Imputation,
    visualize_att_for_one_head_across_time,
    visualize_att_for_target_t_across_heads
)
from inference_full_tile import Dataset_from_files
import json

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

#### Utils functions

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.patches as patches
import ipywidgets as widgets
from IPython.display import display, clear_output


def _to_cpu(x):
    """Copie récursive de tenseurs en CPU (détachés du graph)."""
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().clone()
    elif isinstance(x, dict):
        return {k: _to_cpu(v) for k, v in x.items()}
    elif isinstance(x, (list, tuple)):
        return type(x)(_to_cpu(v) for v in x)
    return x


def sample_to_batch(sample: dict) -> dict:
    batch = {}
    for k, v in sample.items():
        if isinstance(v, torch.Tensor):
            batch[k] = v.unsqueeze(dim=0)
        else:
            batch[k] = v
    return batch


def plot_seq(batch, pred, c_index, BRIGHTNESS_FACTOR=3, t_sampled=None):
    if t_sampled is None:
        images = torch.concatenate(
            [
                batch['y'][0][:, c_index, :, :].swapaxes(1, 3).swapaxes(1, 2),
                batch['x'][0][:, c_index, :, :].swapaxes(1, 3).swapaxes(1, 2),
                pred[0][:, c_index, :, :].swapaxes(1, 3).swapaxes(1, 2),
            ],
            dim=0,
        )
    else:
        images = torch.concatenate(
            [
                batch['y'][0][:, c_index, :, :].swapaxes(1, 3).swapaxes(1, 2)[t_sampled],
                batch['x'][0][:, c_index, :, :].swapaxes(1, 3).swapaxes(1, 2)[t_sampled],
                pred[0][:, c_index, :, :].swapaxes(1, 3).swapaxes(1, 2)[t_sampled],
            ],
            dim=0,
        )
    images = visutils.apply_brightness_factor(images, factor=BRIGHTNESS_FACTOR)
    ncols = int(images.shape[0] / 3)
    fig, axes = plt.subplots(nrows=3, ncols=ncols, figsize=(15, 7))
    column_labels = [f't{i + 1}' for i in range(ncols + 1)]
    for idx, ax in enumerate(axes.flat):
        ax.imshow(images[idx])
        ax.axis('off')
        rect = patches.Rectangle((0, 0), 1, 1, 
                                   transform=ax.transAxes,
                                   fill=False, 
                                   edgecolor='black', 
                                   linewidth=1)
        ax.add_patch(rect)
        if idx < ncols:
            ax.set_title(column_labels[idx], fontsize=12, fontweight='bold')
    plt.subplots_adjust(wspace=0.05, hspace=0.1)
    
    row_labels = ['Target', 'Inputs', 'Predictions']
    for i, label in enumerate(row_labels):
        axes[i, 0].annotate(label, 
                            xy=(-0.1, 0.5),
                            xycoords='axes fraction',
                            fontsize=10, 
                            fontweight='bold',
                            ha='right',
                            va='center',
                            rotation=90)
    
    plt.subplots_adjust(left=0.18)
    plt.show()


def plot_seq_RGB_NIR(batch, pred, n_visible=8):
    # plot R-G-B
    plot_seq_interactive(batch, pred, c_index=[2, 1, 0], BRIGHTNESS_FACTOR=3, n_visible=n_visible)
    # plot NIR-R-G
    plot_seq_interactive(batch, pred, c_index=[6, 2, 1], BRIGHTNESS_FACTOR=2, n_visible=n_visible)

# ── Versions interactives avec slider ──────────────────────────────────────────

def _draw_grid(images, ncols, t_offset, row_labels, ax_array, fig, BRIGHTNESS_FACTOR):
    """Dessine la grille Target / Inputs / Predictions pour un sous-ensemble de timesteps."""
    images = visutils.apply_brightness_factor(images, factor=BRIGHTNESS_FACTOR)
    column_labels = [f't{t_offset + i}' for i in range(ncols)]
    for idx, ax in enumerate(ax_array.flat):
        ax.imshow(images[idx])
        ax.axis('off')
        rect = patches.Rectangle((0, 0), 1, 1,
                                 transform=ax.transAxes,
                                 fill=False, edgecolor='black', linewidth=1)
        ax.add_patch(rect)
        if idx < ncols:
            ax.set_title(column_labels[idx], fontsize=12, fontweight='bold')
    for i, label in enumerate(row_labels):
        ax_array[i, 0].annotate(label, xy=(-0.1, 0.5), xycoords='axes fraction',
                                fontsize=10, fontweight='bold',
                                ha='right', va='center', rotation=90)


def plot_seq_interactive(batch, pred, c_index, BRIGHTNESS_FACTOR=3, n_visible=8):
    """Version interactive de plot_seq avec un slider temporel.
    Les tenseurs sont copiés en CPU pour survivre à la suppression des variables d'origine."""
    batch = _to_cpu(batch)
    pred = _to_cpu(pred)

    target = batch['y'][0][:, c_index, :, :].swapaxes(1, 3).swapaxes(1, 2)
    inputs = batch['x'][0][:, c_index, :, :].swapaxes(1, 3).swapaxes(1, 2)
    preds  = pred[0][:, c_index, :, :].swapaxes(1, 3).swapaxes(1, 2)

    T = target.shape[0]
    n_visible = min(n_visible, T)

    out = widgets.Output()
    slider = widgets.IntSlider(
        value=0, min=0, max=max(T - n_visible, 0), step=1,
        description='t_start :',
        continuous_update=False,
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='80%'),
    )
    label = widgets.Label(value=f'Série totale : {T} dates  |  Fenêtre : {n_visible}')

    def _update(_change=None):
        t0 = slider.value
        idx = slice(t0, t0 + n_visible)
        images = torch.concatenate([target[idx], inputs[idx], preds[idx]], dim=0)
        with out:
            clear_output(wait=True)
            fig, axes = plt.subplots(nrows=3, ncols=n_visible,
                                     figsize=(2.2 * n_visible, 7))
            _draw_grid(images, n_visible, t0,
                       ['Target', 'Inputs', 'Predictions'],
                       axes, fig, BRIGHTNESS_FACTOR)
            plt.subplots_adjust(left=0.10, wspace=0.05, hspace=0.1)
            plt.show()

    slider.observe(_update, names='value')
    display(widgets.VBox([label, slider, out]))
    _update()


def plot_seq_RGB_NIR_interactive(
    targets, 
    inputs,
    preds,
    n_visible=8):
    """Version interactive de plot_seq_RGB_NIR avec un slider partagé RGB / NIR.
    Les tenseurs sont copiés en CPU pour survivre à la suppression des variables d'origine."""
    targets = _to_cpu(targets)
    inputs = _to_cpu(inputs)
    preds = _to_cpu(preds)

    target_rgb = targets[:, [2, 1, 0], :, :].swapaxes(1, 3).swapaxes(1, 2)
    inputs_rgb = inputs[:, [2, 1, 0], :, :].swapaxes(1, 3).swapaxes(1, 2)
    preds_rgb  = preds[:, [2, 1, 0], :, :].swapaxes(1, 3).swapaxes(1, 2)

    target_nir = targets[:, [6, 2, 1], :, :].swapaxes(1, 3).swapaxes(1, 2)
    inputs_nir = inputs[:, [6, 2, 1], :, :].swapaxes(1, 3).swapaxes(1, 2)
    preds_nir  = preds[:, [6, 2, 1], :, :].swapaxes(1, 3).swapaxes(1, 2)

    T = target_rgb.shape[0]
    n_visible = min(n_visible, T)

    out = widgets.Output()
    slider = widgets.IntSlider(
        value=0, min=0, max=max(T - n_visible, 0), step=1,
        description='t_start :',
        continuous_update=False,
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='80%'),
    )
    label = widgets.Label(value=f'Série totale : {T} dates  |  Fenêtre : {n_visible}')

    def _update(_change=None):
        t0 = slider.value
        idx = slice(t0, t0 + n_visible)

        imgs_rgb = torch.concatenate([target_rgb[idx], inputs_rgb[idx], preds_rgb[idx]], dim=0)
        imgs_nir = torch.concatenate([target_nir[idx], inputs_nir[idx], preds_nir[idx]], dim=0)

        row_labels = ['Target', 'Inputs', 'Predictions']
        with out:
            clear_output(wait=True)
            fig, axes = plt.subplots(nrows=6, ncols=n_visible,
                                     figsize=(2.2 * n_visible, 13))
            _draw_grid(imgs_rgb, n_visible, t0, row_labels,
                       axes[:3], fig, BRIGHTNESS_FACTOR=3)
            _draw_grid(imgs_nir, n_visible, t0, row_labels,
                       axes[3:], fig, BRIGHTNESS_FACTOR=2)

            # Titres de section
            fig.text(0.02, 0.78, 'RGB', fontsize=14, fontweight='bold',
                     rotation=90, va='center')
            fig.text(0.02, 0.35, 'NIR-R-G', fontsize=14, fontweight='bold',
                     rotation=90, va='center')

            plt.subplots_adjust(left=0.08, wspace=0.05, hspace=0.15)
            plt.show()

    slider.observe(_update, names='value')
    display(widgets.VBox([label, slider, out]))
    _update()

def getitem_from_mgrsc(df, mgrsc, window: str = None):
    if window is None:
        return df[df["mgrs25"] == mgrsc].index.values
    else:
        return df[(df["mgrs25"] == mgrsc) & (df["window"] == window)].index.values[0]

## Inférences faites depuis le fichier hdf5 

In [3]:
config_file = Path("./configs/config_run_train.yaml")
default_config_path = Path("./configs/default.yaml")
cfg_custom = config_utils.read_config(config_file)
cfg_default = config_utils.read_config(default_config_path)
config = OmegaConf.merge(cfg_default, cfg_custom)

# Faire différents changments dans les fichiers de configs pour les vérifications
config.data.hdf5_file = "/mnt/DATA_10T/data_rpg/circa/hdf5/CIRCA_CR_merged.hdf5"
config.mask.mask_type = 'random_fully_masked'
config.data.max_seq_length = None
config.training_settings.batch_size = 1
config.mask.ratio_masked_frames = 0.8
config.mask.intersect_real_cloud_masks = False
config.mask.dilate_cloud_masks = False

BRIGHTNESS_FACTOR = 3
phase = "test"

path_inference_config_file = Path("/home/SPeillet/rpg_3STR/cloud_reconstruction/U-TILISE/configs/config_run_eval.yaml")
# path_ckpt_config_file = Path("/mnt/DATA_10T/data_rpg/outputs/U-TILISE/results/ALL_SAR_120_epochs_2025-07-11_16-56/config.yaml")
# path_ckpt_pth = Path("/mnt/DATA_10T/data_rpg/outputs/U-TILISE/results/ALL_SAR_120_epochs_2025-07-11_16-56/checkpoints/Model_best.pth")

#
# path_ckpt_config_file = Path("/home/SPeillet/data_rpg/outputs/U-TILISE/results/multistream_all_bands_random_clouds/seq_length_12_bs_4_acc_iter_2_da_masked_0.6_fully_masked_0.2_epochs_65/config.yaml")
# path_ckpt_config_file = Path("/home/SPeillet/data_rpg/outputs/U-TILISE/results/multistream_all_bands_random_clouds/seq_length_12_bs_4_acc_iter_2_da_masked_0.6_fully_masked_0.2_epochs_65/checkpoints/Model_best.pth")

# path_ckpt_config_file = Path("/home/SPeillet/data_rpg/outputs/U-TILISE/results/all_bands_sar_closest_mix_da/config.yaml")
# path_ckpt_pth = Path("/home/SPeillet/data_rpg/outputs/U-TILISE/results/all_bands_sar_closest_mix_da/checkpoints/Model_best.pth")

path_ckpt_config_file = Path("/home/SPeillet/data_rpg/outputs/U-TILISE/results/all_bands_sar_closest_mix_random_fully_masked_da/2026-03-14_10-59/config.yaml")
path_ckpt_pth = Path("/home/SPeillet/data_rpg/outputs/U-TILISE/results/all_bands_sar_closest_mix_random_fully_masked_da/2026-03-14_10-59/checkpoints/Model_best.pth")

training_config_file = config_utils.read_config(path_ckpt_config_file)
inference_config_file = config_utils.read_config(path_inference_config_file)

temporal_window = 6
# Faire la différence entre l'imputation d'une TS (avec plusieurs intervalles de dates) et plusieurs observations.
inference_imputation = Imputation(
    config_file_train=path_inference_config_file, # Config permettant de faire la configuration de l'inference
    method="utilise",
    checkpoint=path_ckpt_pth,
    config_file_test=path_ckpt_config_file, # Fichier ayant servi à l'entrainement du modèle (update params model)
    temporal_window=temporal_window,
)

dset = data_utils.get_dataset(config, phase=phase)

# dset = torch.utils.data.Subset(dset, range(SUBSET_LENGTH))
dataloader = torch.utils.data.DataLoader(
    dataset=dset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    collate_fn=None,
    pin_memory=False,
    drop_last=False,
)



Loaded transforms from ./data/CIRCA_patches_datasets_with_transforms.json.
Checkpoint '/home/SPeillet/data_rpg/outputs/U-TILISE/results/all_bands_sar_closest_mix_random_fully_masked_da/2026-03-14_10-59/checkpoints/Model_best.pth' loaded.
Chosen epoch: 24



In [4]:
item = 500
t_sampled = [2, 4, 6, 8, 9, 10, 13]
t_masked = {"indices_masked": [1, 2, 4, 6]}
# t_sampled = [34, 35, 36, 37, 38, 39, 40, 41, 42]
# t_masked = None

# Faire différents changments dans les fichiers de configs pour les vérifications
config.data.hdf5_file = "/mnt/DATA_10T/data_rpg/circa/hdf5/CIRCA_CR_merged.hdf5"
config.mask.mask_type = 'random_fully_masked'
config.data.max_seq_length = None
config.training_settings.batch_size = 1
config.mask.ratio_masked_frames = 0.5
config.mask.intersect_real_cloud_masks = False
config.mask.dilate_cloud_masks = False

BRIGHTNESS_FACTOR = 3

dl1 = torch.utils.data.DataLoader(
    dataset=data_utils.get_dataset(config, phase="test"),
    batch_size=1,
    shuffle=False,
    num_workers=0,
    collate_fn=None,
    pin_memory=False,
    drop_last=False,
)

sample1 = dl1.dataset.__getitem__(item, t_sampled=t_sampled, t_masked=t_masked)
print(sample1["info"])
batch1 = sample_to_batch(sample1)

batch1, y_pred, att = inference_imputation.impute_sample(
    batch=batch1,
    return_att=True,
    # t_start=t_start,
    # t_end=t_end,
)

# plot_seq_RGB_NIR(batch1, y_pred)
plot_seq_RGB_NIR_interactive(
targets=batch1["y"][0],
inputs=batch1["x"][0],
preds=y_pred[0],
) 

{'mgrs': '30UXV', 'mgrs25': '30UXV_row-2_col-2', 'window': '1536_1940_256_256'}


## Inférence faite depuis les fichiers du store-dai

### Good sample

In [5]:
store_dai = Path("/mnt/stores/store_dai")
path_dataset_circa = store_dai / "projets/pac/3str/EXP_2/Data_Raster"
data_optique = path_dataset_circa / "optique_dataset"
data_radar = path_dataset_circa / "radar_dataset_v4"
path_test_set_mgrs25 = store_dai / "projets/pac/3str/EXP_2/train_val_test/MGRSC_test.json"
test_mgrs25 = json.load(open(path_test_set_mgrs25))
# Faire différents changments dans les fichiers de configs pour les vérifications
config.data.hdf5_file = "/mnt/DATA_10T/data_rpg/circa/hdf5/CIRCA_CR_merged.hdf5"
config.mask.mask_type = 'random_fully_masked'
config.data.max_seq_length = None
config.training_settings.batch_size = 1
config.mask.ratio_masked_frames = 0.0
config.mask.intersect_real_cloud_masks = False
config.mask.dilate_cloud_masks = False

BRIGHTNESS_FACTOR = 3

dict_mgrs = {f.stem: f for f in data_optique.iterdir()}
dict_mgrsc = {p.stem: p for f in dict_mgrs.values() for p in f.iterdir()}

mgrsc_good_sample = '30UXV_row-2_col-2'
window_good_sample = (1536, 1940, 256, 256)
image_size = [256, 256]
OVERLAP = 0

ds_from_files = Dataset_from_files(
    mgrsc=mgrsc_good_sample,
    data_optique=data_optique,
    data_radar=data_radar,
    image_size=image_size,
    overlap=OVERLAP,
    fill_value=1.0,
    mask_type='original_masks',
    load_dataset="./tiles_windows.csv",
)

dl_from_files = torch.utils.data.DataLoader(
    dataset=ds_from_files,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    collate_fn=None,
    pin_memory=False,
    drop_last=False,
)

In [6]:
index_good_sample = getitem_from_mgrsc(ds_from_files.mgrsc_dataset, mgrsc=mgrsc_good_sample, window=window_good_sample)
item = index_good_sample
# t_sampled = None
t_sampled = None
t_masked = None

sample_from_files = dl_from_files.dataset.__getitem__(item, t_sampled=[19, 20, 21, 22, 64, 65, 75]) #29
batch_from_files = sample_to_batch(sample_from_files)
# batch_from_files["position_days"] = batch_hdf5["position_days"]
# batch_from_files["masks_valid_obs"] = batch_hdf5["masks_valid_obs"]
batch_from_files, y_pred_from_files, att = inference_imputation.impute_sample(
    batch=batch_from_files,
    return_att=True,
    # t_start=t_start,
    # t_end=t_end,
)

plot_seq_RGB_NIR_interactive(
    targets=batch_from_files["y"][0],
    inputs=batch_from_files["x"][0],
    preds=y_pred_from_files[0],
    n_visible=8,
)

Original masks shape: torch.Size([7, 2, 256, 256])


In [7]:
sample_from_files = dl_from_files.dataset.__getitem__(item, t_sampled=t_sampled)
batch_from_files = sample_to_batch(sample_from_files)

batch_from_files, y_pred_from_files, att = inference_imputation.impute_sample(
    batch=batch_from_files,
    return_att=True,
    # t_start=t_start,
    # t_end=t_end,
)

plot_seq_RGB_NIR_interactive(
    targets=batch_from_files["y"][0], 
    inputs=batch_from_files["x"][0],
    preds=y_pred_from_files[0],
    n_visible=8,
)

KeyboardInterrupt: 

In [ ]:
batch_from_files, y_pred_from_files, att = inference_imputation.impute_sample(
    batch=batch_from_files,
    return_att=True,
    # t_start=t_start,
    # t_end=t_end,
)

plot_seq_RGB_NIR_interactive(
    targets=batch_from_files["y"][0],
    inputs=batch_from_files["x"][0],
    preds=y_pred_from_files[0],
    n_visible=8,
)

In [ ]:
sample_from_files = dl_from_files.dataset.__getitem__(item, t_sampled=[19, 20, 64, 65, 75]) #29
batch_from_files = sample_to_batch(sample_from_files)

Original masks shape: torch.Size([5, 2, 256, 256])


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def analyze_and_plot_distributions(b_hdf5, b_tif):
    """
    Calcule et compare les distributions de pixels par grandes catégories
    (Optique S2 et potentiellement Radar S1 si présent) entre les deux approches.
    Rejette les pixels masqués pour ne juger que de la vraie donnée.
    """
    
    # 1. Extraction et nettoyage de 'y' (cibles complètes)
    # y est au format (Batch, Temps, Canaux, H, W). Pour la distribution globale on s'en fiche du temps et x,y.
    # On va regrouper tous les pixels valides.
    
    y_h5  = b_hdf5["y"][0].detach().cpu().numpy()
    y_tif = b_tif["y"][0].detach().cpu().numpy()

    c_s2 = 10 # Nombre de canaux Sentinel-2
    # Séparation Optique S2 (Les 10 premiers canaux)
    s2_h5  = y_h5[:, :c_s2, :, :].flatten()
    s2_tif = y_tif[:, :c_s2, :, :].flatten()
    
    print("="*60)
    print("STATISTIQUES GLOBALES - IMAGES CIBLES (y)")
    print("="*60)
    print(f"{'Source':<15} | {'Min':<8} | {'Max':<8} | {'Moyenne':<8} | {'Médiane':<8} | {'Ecart-type':<8}")
    print("-" * 60)
    print(f"{'HDF5 (S2)':<15} | {s2_h5.min():.4f}   | {s2_h5.max():.4f}   | {s2_h5.mean():.4f}   | {np.median(s2_h5):.4f}   | {s2_h5.std():.4f}")
    print(f"{'TIF_file (S2)':<15} | {s2_tif.min():.4f}   | {s2_tif.max():.4f}   | {s2_tif.mean():.4f}   | {np.median(s2_tif):.4f}   | {s2_tif.std():.4f}")
    
    # Si le radar est présent (Nb Canaux > 10)
    has_sar = y_h5.shape[1] > c_s2
    if has_sar:
        s1_h5 = y_h5[:, c_s2:, :, :].flatten()
        s1_tif = y_tif[:, c_s2:, :, :].flatten()
        print(f"{'HDF5 (S1_SAR)':<15} | {s1_h5.min():.4f}   | {s1_h5.max():.4f}   | {s1_h5.mean():.4f}   | {np.median(s1_h5):.4f}   | {s1_h5.std():.4f}")
        print(f"{'TIF_file(S1_SAR)':<15} | {s1_tif.min():.4f}   | {s1_tif.max():.4f}   | {s1_tif.mean():.4f}   | {np.median(s1_tif):.4f}   | {s1_tif.std():.4f}")
        
    print("\n")
        
    # 2. Visualisation des distributions de densité (Histogrammes)
    fig, axes = plt.subplots(1, 2 if has_sar else 1, figsize=(12 if has_sar else 6, 5))
    
    if not isinstance(axes, np.ndarray):
        axes = [axes]
        
    # Hist Optique
    # On limite à 1.0 au cas où, pour éviter que des valeurs extrêmes (outliers) écrasent le visuel
    bins_S2 = np.linspace(0, min(1.0, max(s2_h5.max(), s2_tif.max())), 100) 
    
    axes[0].hist(s2_h5, bins=bins_S2, alpha=0.5, density=True, label='HDF5 Pipeline', color='blue')
    axes[0].hist(s2_tif, bins=bins_S2, alpha=0.5, density=True, label='GeoTIFF Pipeline', color='orange')
    axes[0].set_title('Densité des valeurs des pixels Sentinel-2')
    axes[0].set_xlabel('Valeur normalisée (0 à 1)')
    axes[0].set_ylabel('Densité')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Hist Radar si applicable
    if has_sar:
        bins_S1 = np.linspace(0, 1.0, 100)
        axes[1].hist(s1_h5, bins=bins_S1, alpha=0.5, density=True, label='HDF5 Pipeline', color='blue')
        axes[1].hist(s1_tif, bins=bins_S1, alpha=0.5, density=True, label='GeoTIFF Pipeline', color='orange')
        axes[1].set_title('Densité des valeurs des pixels Sentinel-1 (SAR)')
        axes[1].set_xlabel('Valeur normalisée (0 à 1)')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# Exécution du test de compatibilité distributionnelle
analyze_and_plot_distributions(batch_hdf5, batch_from_files)

NameError: name 'batch_hdf5' is not defined

In [ ]:
batch_hdf5.keys()

In [ ]:
batch_hdf5["position_days"]

In [ ]:
batch_hdf5["position_days"]

### Observation direct des inférences en mode produit

In [ ]:
from rasterio.windows import Window
from dataloader_CIRCA.tools.mask_generation import masks_init_filling
import torch

def read_data_by_window(
    target_window,
    path_file,
):
    """
        Return a ndarray of a crop tile with shape T * H * W * C
    """
    with rasterio.open(path_file) as src:
        array = src.read(window=Window(*target_window))
        array = array.reshape((int(array.shape[0] // 12), 12, array.shape[1], array.shape[2])).astype(np.float32)
        return array[:, :10, ...], array[:, -1, ...]

In [ ]:
mgrsc_target = '30UXV_row-2_col-2'
target_window = (1536, 1940, 256, 256)
# mgrsc_target = mgrsc_bad_sample = "30TYS_row-4_col-4"
# target_window = window_bad_sample = (1536, 512, 256, 256)

path_preds = store_dai / "tmp/speillet/inferences"
pred_file = path_preds / f"pred_mgrsc_{mgrsc_target}.tif"
assert pred_file.exists(), f"File {pred_file} doesn't exists.."
preds, _ = read_data_by_window(
    target_window,
    path_file=pred_file,
)

path_inputs = store_dai / "projets/pac/3str/EXP_2/Data_Raster/optique_dataset"
input_file = path_inputs / mgrsc_target[:5] / f"MGRS25-{mgrsc_target}" /  f"bands_stacked_{mgrsc_target}.tif"
assert input_file.exists(), f"File {input_file} doesn't exists.."
inputs, cloud_mask = read_data_by_window(
    target_window,
    path_file=input_file,
)

inputs, preds, cloud_mask = torch.from_numpy(inputs), torch.from_numpy(preds), torch.from_numpy(cloud_mask)
inputs_masked, masks = masks_init_filling(
    seq=inputs.clone(),
    masks=cloud_mask.clone(),
    fill_type="fill_value",
    fill_value=1,
    dilate_cloud_masks=False,
)

In [ ]:
plot_seq_RGB_NIR_interactive(
    targets=inputs/10000, 
    inputs=inputs_masked/10000,
    preds=preds/10000,
    n_visible=8,
)

### Inférence depuis le hdf5

In [ ]:
import random
from functools import partial
from omegaconf import DictConfig
import os
import torch
import rasterio
import numpy as np
from pathlib import Path
import h5py
from typing import Dict, Any
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
from lib import config_utils
from lib import data_utils
from lib import visutils # gallery / apply_brightness_factor / sequence2gallery 

from lib.eval_tools import (
    Imputation,
    visualize_att_for_one_head_across_time,
    visualize_att_for_target_t_across_heads
)

from dataloader_CIRCA.datasets import CIRCA_ADAPTED2UTILISE_Dataset
from lib.data_utils import pad_collate

from typing import Any, Dict, Literal
import math
import torch
from prodict import Prodict
from torch import Tensor
from lib.data_utils import extract_sample
from lib import config_utils
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)   # ou FutureWarning, UserWarning, ...

In [ ]:
def sample_to_batch(sample: dict) -> dict:
    batch = {}
    for k, v in sample.items():
        if isinstance(v, torch.Tensor):
            batch[k] = v.unsqueeze(dim=0)
        else:
            batch[k] = v
    return batch

In [ ]:
item = 500
# t_sampled = [2, 4, 6, 8, 9, 10, 13]
# t_masked = {"indices_masked": [1, 2, 4, 6]}

t_sampled = [0, 1, 2, 3, 4, 5, 6]
t_masked = {"indices_masked": [1, 2, 3, 4, 5]}

phase = "test"
BRIGHTNESS_FACTOR = 3
# Setup configuration
config_file = Path("./configs/config_run_train.yaml")
default_config_path = Path("./configs/default.yaml")
cfg_custom = config_utils.read_config(config_file)

cfg_default = config_utils.read_config(default_config_path)
config = OmegaConf.merge(cfg_default, cfg_custom)

config.data.hdf5_file = "/mnt/DATA_10T/data_rpg/circa/hdf5/CIRCA_CR_merged.hdf5"
config.mask.mask_type = 'random_fully_masked'
config.data.max_seq_length = None
config.training_settings.batch_size = 1
config.mask.ratio_masked_frames = 0.5
config.mask.intersect_real_cloud_masks = False
config.mask.dilate_cloud_masks = False

BRIGHTNESS_FACTOR = 3

dl_hdf5 = torch.utils.data.DataLoader(
    dataset=data_utils.get_dataset(config, phase="test"),
    batch_size=1,
    shuffle=False,
    num_workers=0,
    collate_fn=None,
    pin_memory=False,
    drop_last=False,
)

sample_hdf5 = dl_hdf5.dataset.__getitem__(item, t_sampled=t_sampled, t_masked=t_masked)
print(sample_hdf5["info"])

In [ ]:
batch_hdf5 = sample_to_batch(sample_hdf5)
batch_hdf5["position_days"] = batch_from_files["position_days"]
batch_hdf5, y_pred_hdf5, att = inference_imputation.impute_sample(
    batch=batch_hdf5,
    return_att=True,
    # t_start=t_start,
    # t_end=t_end,
)

plot_seq_RGB_NIR_interactive(
    targets=batch_hdf5["y"][0],
    inputs=batch_hdf5["x"][0],
    preds=y_pred_hdf5[0],
    n_visible=8,
)

In [ ]:
batch_from_files, y_pred_from_files, att = inference_imputation.impute_sample(
    batch=batch_from_files,
    return_att=True,
    # t_start=t_start,
    # t_end=t_end,
)

plot_seq_RGB_NIR_interactive(
    targets=batch_from_files["y"][0], 
    inputs=batch_from_files["x"][0],
    preds=y_pred_from_files[0],
    n_visible=8,
)

In [ ]:
batch_from_files["masks_valid_obs"]

In [ ]:
batch_hdf5["masks_valid_obs"]

In [ ]:
def stats(tensor):
    return f"Min={tensor.min().item():.3f}, Max={tensor.max().item():.3f}, Mean={tensor.float().mean().item():.3f}, Uniques={len(tensor.unique())}"

print("=== STATISTIQUES DES TENSEURS CRITIQUES ===")
for k in ["x", "y", "masks", "cloud_mask", "position_days", "days"]:
    print(f"\n--- {k.upper()} ---")
    print(f"HDF5 : {stats(batch_hdf5[k])}")
    print(f"TIF  : {stats(batch_from_files[k])}")
    if k in ["masks", "cloud_mask", "position_days", "days"]:
        print(f"HDF5 Jours/Vals : {batch_hdf5[k].flatten()[:10].tolist()}")
        print(f"TIF  Jours/Vals : {batch_from_files[k].flatten()[:10].tolist()}")